<a href="https://colab.research.google.com/github/korzhimanov/dsp-seminars/blob/main/seminars/2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическое задание №2
## Амплитудная и фазовая модуляция: генерация, спектры, демодуляция и устойчивость к шумам

**Цель работы:**
- Научиться генерировать сигналы с амплитудной (АМ) и фазовой (ФМ) модуляцией.
- Исследовать спектральные характеристики модулированных сигналов.
- Изучить влияние индекса модуляции на спектр ФМ сигнала.
- Освоить демодуляцию АМ и ФМ с помощью преобразования Гильберта и демодуляцию ФМ.
- Оценить устойчивость АМ и ФМ к амплитудным и фазовым шумам на примере реального аудиосигнала.

**Необходимое ПО:** Python 3, библиотеки `numpy`, `matplotlib`, `scipy`, `ipywidgets`, `IPython.display`. Работа выполняется в Jupyter Notebook / Google Colab.

**Формат сдачи:** один Jupyter Notebook с кодом, графиками и текстовыми выводами. Имя файла: `Seminar2_Фамилия.ipynb`.

## Часть 1. Генерация и визуализация модулированных сигналов

### 1.1. Импорт библиотек и параметры

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io import wavfile
from scipy.signal import hilbert
import ipywidgets as widgets
from IPython.display import Audio, display

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)

# Параметры дискретизации для демонстрации
fs = 10000                     # частота дискретизации (Гц)
T = 0.1                        # длительность сигнала (с)
t = np.linspace(0, T, int(fs*T), endpoint=False)

# Несущая
fc = 1000                      # частота несущей (Гц)
Ac = 1.0                       # амплитуда несущей

### 1.2. Модулирующий сигнал

В качестве модулирующего сигнала используйте сумму двух синусоид с частотами 100 Гц и 200 Гц, нормализованную к диапазону [-1, 1].

In [ ]:
fm1 = 100
fm2 = 200
m_t = 0.8 * (np.sin(2*np.pi*fm1*t) + np.sin(2*np.pi*fm2*t)) / 2
m_t = m_t / np.max(np.abs(m_t))  # теперь от -1 до 1

plt.figure()
plt.plot(t, m_t)
plt.title('Модулирующий сигнал (100 Гц + 200 Гц)')
plt.xlabel('Время (с)')
plt.ylabel('Амплитуда')
plt.grid(True)
plt.show()

### 1.3. Амплитудная модуляция (АМ)

Формула АМ:
$$s_{AM}(t) = A_c [1 + a_{AM}\cdot m(t)] \cos(2\pi f_c t)
$$
Параметр $a_{AM}$ регулирует _глубину модуляции_, он не может быть больше 1 (при $a_am>1$ и исходном сигнале $m(t)$, нормированном на единицу, амплитуда результирующего сигнала в некоторые промежутки времени становится отрицательной).

In [ ]:
a_am = 0.2
s_am = Ac * (1 + a_am * m_t) * np.cos(2 * np.pi * fc * t)

plt.figure()
plt.plot(t, s_am)
plt.title('АМ сигнал (несущая 1000 Гц)')
plt.xlabel('Время (с)')
plt.ylabel('Амплитуда')
plt.grid(True)
plt.show()

### 1.4. Фазовая модуляция (ФМ)

Формула ФМ:
$$
s_{PM}(t) = A_c \cos(2\pi f_c t + \beta \cdot m(t)),
$$

где $\beta$ – индекс фазовой модуляции.

In [ ]:
beta_pm = 5
s_pm = Ac * np.cos(2 * np.pi * fc * t + beta_pm * m_t)

plt.figure()
plt.plot(t, s_pm)
plt.title(f'ФМ сигнал (β = {beta_pm})')
plt.xlabel('Время (с)')
plt.ylabel('Амплитуда')
plt.grid(True)
plt.show()

**Задание 1.1.** Постройте на одном графике модулирующий сигнал и оба модулированных сигнала (АМ, ФМ). Опишите визуальные отличия во временной форме.

## Часть 2. Спектральный анализ модулированных сигналов

### 2.1. Функция построения амплитудного спектра

In [ ]:
def plot_spectrum(s, fs, title):
    N = len(s)
    X = np.fft.fft(s)
    freq = np.fft.fftfreq(N, 1/fs)[:N//2]
    mag = np.abs(X[:N//2]) / (N//2)
    plt.stem(freq, mag, basefmt=" ")
    plt.title(title)
    plt.xlabel('Частота (Гц)')
    plt.ylabel('Амплитуда')
    plt.grid(True)
    plt.xlim(0, 2000)
    plt.show()

### 2.2. Спектр АМ

In [ ]:
plot_spectrum(s_am, fs, 'АМ спектр (β = 1 по глубине)')

### 2.3. Спектр ФМ при разных индексах модуляции

In [ ]:
plot_spectrum(s_pm, fs, f'ФМ спектр (β = {beta_pm})')

**Задание 2.1.** Для ФМ измените индекс модуляции (например, β = 1, 2, 5, 10) и постройте спектры. Опишите, как меняется ширина спектра и количество боковых полос.

### 2.4. Интерактивное исследование спектра ФМ

In [ ]:
def pm_spectrum(beta=5):
    s = Ac * np.cos(2 * np.pi * fc * t + beta * m_t)
    N = len(s)
    X = np.fft.fft(s)
    freq = np.fft.fftfreq(N, 1/fs)[:N//2]
    mag = np.abs(X[:N//2]) / (N//2)
    plt.stem(freq, mag, basefmt=" ")
    plt.title(f'ФМ спектр при β={beta}')
    plt.xlabel('Частота (Гц)')
    plt.ylabel('Амплитуда')
    plt.xlim(0, 2000)
    plt.grid(True)
    plt.show()

interact(pm_spectrum, beta=widgets.FloatSlider(min=0.1, max=20, step=0.5, value=5));

**Вопросы:**
- Почему спектр АМ содержит только три ярко выраженных компоненты (несущая и две боковых)?
- Почему спектр ФМ при больших β расширяется и появляется множество боковых полос?

## Часть 3. Демодуляция сигналов

### 3.1. Демодуляция АМ с помощью преобразования Гильберта

Преобразование Гильберта позволяет получить аналитический сигнал, огибающая которого содержит исходное сообщение.

In [ ]:
# Аналитический сигнал для АМ
analytic_am = hilbert(s_am)
envelope = np.abs(analytic_am)          # огибающая = 1 + m(t)

# Восстановленное сообщение (убираем постоянную составляющую)
demod_am = envelope - 1

# Сравнение с исходным
plt.figure()
plt.plot(t, m_t, label='Исходное сообщение')
plt.plot(t, demod_am, '--', label='Демодулированное (АМ)')
plt.xlabel('Время (с)')
plt.ylabel('Амплитуда')
plt.title('Демодуляция АМ сигнала')
plt.legend()
plt.grid(True)
plt.show()

**Задание 3.1.** Вычислите среднеквадратичную ошибку между исходным и восстановленным сообщением. Почему они не совпадают идеально? (Подсказка: краевые эффекты преобразования Гильберта, конечность окна).

### 3.2. Демодуляция ФМ сигнала

Для ФМ информация заключена в отклонении фазы от линейного тренда. Используем аналитический сигнал и выделим мгновенную фазу.

In [ ]:
# Аналитический сигнал для ФМ
analytic_pm = hilbert(s_pm)
inst_phase = np.unwrap(np.angle(analytic_pm))   # развернутая фаза

# Убираем линейную составляющую (фаза несущей)
phase_no_carrier = inst_phase - 2 * np.pi * fc * t

# Восстановленное сообщение (делим на индекс модуляции)
demod_pm = phase_no_carrier / beta_pm

# Сравнение с исходным
plt.figure()
plt.plot(t, m_t, label='Исходное сообщение')
plt.plot(t, demod_pm, '--', label='Демодулированное (ФМ)')
plt.xlabel('Время (с)')
plt.ylabel('Амплитуда')
plt.title('Демодуляция ФМ сигнала')
plt.legend()
plt.grid(True)
plt.show()

**Вопрос:** Почему важно использовать `np.unwrap` для фазы? Что произойдёт при скачках фазы?

## Часть 4. Устойчивость к шумам (на примере аудиосигнала)

В этой части мы работаем с реальным аудиофайлом. Можно загрузить любой WAV-файл (речь, музыка) или использовать сгенерированный тестовый сигнал.

### 4.1. Загрузка аудио (или генерация тестового сигнала)

In [ ]:
from google.colab import files
uploaded = files.upload()   # выберите файл .wav

In [ ]:
fs_audio = 8000
t_audio = np.linspace(0, 2, 2*fs_audio, endpoint=False)
melody = (np.sin(2*np.pi*440*t_audio) + 
          np.sin(2*np.pi*523*t_audio) + 
          np.sin(2*np.pi*659*t_audio)) / 3
melody = melody / np.max(np.abs(melody))

print("Исходное аудио:")
display(Audio(melody, rate=fs_audio))

### 4.2. Подготовка к модуляции (интерполяция)

Чтобы несущая частота не перекрывалась с аудиоспектром, повысим частоту дискретизации и выберем несущую выше 4 кГц.

In [ ]:
fs_new = 20000
t_new = np.linspace(0, len(melody)/fs_audio, int(len(melody)*fs_new/fs_audio), endpoint=False)

from scipy import interpolate
f_interp = interpolate.interp1d(np.linspace(0, len(melody)/fs_audio, len(melody)), melody, kind='linear')
melody_resampled = f_interp(t_new)

fc_high = 5000   # несущая 5 кГц
Ac = 1

### 4.3. Модуляция аудиосигнала

In [ ]:
# АМ
s_am_audio = Ac * (1 + 0.8 * melody_resampled) * np.cos(2 * np.pi * fc_high * t_new)

# ФМ (индекс модуляции выберем так, чтобы полоса была разумной)
beta_pm_audio = 5
s_pm_audio = Ac * np.cos(2 * np.pi * fc_high * t_new + beta_pm_audio * melody_resampled)

### 4.4. Добавление шумов

**Амплитудный шум (для АМ):** мультипликативный гауссов шум.

In [ ]:
noise_amp = 0.2 * np.random.randn(len(t_new))
s_am_noisy = s_am_audio * (1 + noise_amp)

**Фазовый шум (для ФМ):** аддитивный гауссов шум к фазе.

In [ ]:
phase_noise = 0.2 * np.random.randn(len(t_new))
s_pm_noisy = s_pm_audio * (1 + noise_amp)

### 4.5. Демодуляция зашумленных сигналов

**Демодуляция АМ (Гильберт):**

In [ ]:
analytic_am_noisy = hilbert(s_am_noisy)
envelope_noisy = np.abs(analytic_am_noisy)
demod_am_noisy = envelope_noisy - 1   # убираем постоянную сотсавляющую

# Интерполяция обратно к исходной частоте дискретизации
demod_am_resampled = np.interp(np.linspace(0, len(t_new)/fs_new, len(melody)), t_new, demod_am_noisy)
demod_am_resampled = demod_am_resampled / np.max(np.abs(demod_am_resampled))

print("АМ после шума и демодуляции:")
display(Audio(demod_am_resampled, rate=fs_audio))

**Демодуляция ФМ с амплитудными шумами:**

In [ ]:
analytic_pm_noisy = hilbert(s_pm_noisy)
inst_phase_noisy = np.unwrap(np.angle(analytic_pm_noisy))
phase_no_carrier_noisy = inst_phase_noisy - 2 * np.pi * fc_high * t_new
demod_pm_noisy = phase_no_carrier_noisy / beta_pm_audio
demod_pm_noisy = demod_pm_noisy - np.mean(demod_pm_noisy)   # убираем постоянное смещение

demod_pm_resampled = np.interp(np.linspace(0, len(t_new)/fs_new, len(melody)), t_new, demod_pm_noisy)
demod_pm_resampled = demod_pm_resampled / np.max(np.abs(demod_pm_resampled))

print("ФМ после шума и демодуляции:")
display(Audio(demod_pm_resampled, rate=fs_audio))

### 4.6. Сравнение спектров

Построим спектры исходного и восстановленных сигналов.

In [ ]:
def plot_spectrum_audio(x, fs, title):
    N = len(x)
    X = np.fft.rfft(x)
    freq = np.fft.rfftfreq(N, 1/fs)
    mag = np.abs(X) / N
    plt.plot(freq, mag)
    plt.title(title)
    plt.xlabel('Частота (Гц)')
    plt.ylabel('Амплитуда')
    plt.grid(True)
    plt.xlim(0, 4000)

plt.figure(figsize=(14,10))

plt.subplot(3,1,1)
plot_spectrum_audio(melody, fs_audio, 'Спектр исходного аудио')

plt.subplot(3,1,2)
plot_spectrum_audio(demod_am_resampled, fs_audio, 'Спектр после АМ + шум + демодуляция')

plt.subplot(3,1,3)
plot_spectrum_audio(demod_pm_resampled, fs_audio, 'Спектр после ФМ + шум + демодуляция')

plt.tight_layout()
plt.show()

 Для каждого вида модуляции:
- Прослушайте восстановленные сигналы и опишите характер искажений (шум, потеря высоких частот, нелинейные искажения).
- Объясните, почему амплитудный шум по-разному влияет на АМ и ФМ.
- Почему фазовый шум практически не слышен в АМ, но сильно искажает ФМ?